In [1]:
import sys
import os
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

import numpy as np
import pandas as pd
import json
import joblib
from src.feature_extractor import FeaturePipeline

model = joblib.load('data/models/baseline_random_forest.pkl')
with open('data/processed/feature_names.json') as f:
    feature_names = json.load(f)
pipeline = FeaturePipeline()
X_train = np.load('data/processed/X_train.npy')
y_train = np.load('data/processed/y_train.npy')
X_test = np.load('data/processed/X_test.npy')
y_test = np.load('data/processed/y_test.npy')

print("Herşey yüklendi!")

Herşey yüklendi!


In [2]:
df = pd.read_csv('data/processed/raw_dataset.csv')
mesru = df[df['label'] == 0]['url']
phishing = df[df['label'] == 1]['url']

print("MEVCUT DATASET DURUMU:")
print("-" * 40)
print(f"Toplam URL: {len(df)}")
print(f"\nMEŞRU ({len(mesru)} toplam):")
print(f"  https://  : {mesru.str.startswith('https://').sum()}")
print(f"  http://   : {mesru.str.startswith('http://').sum()}")
print(f"  Protokolsüz: {(~mesru.str.startswith('http')).sum()}")
print(f"\nPHISHING ({len(phishing)} toplam):")
print(f"  https://  : {phishing.str.startswith('https://').sum()}")
print(f"  http://   : {phishing.str.startswith('http://').sum()}")
print(f"  Protokolsüz: {(~phishing.str.startswith('http')).sum()}")

MEVCUT DATASET DURUMU:
----------------------------------------
Toplam URL: 50000

MEŞRU (25000 toplam):
  https://  : 11634
  http://   : 104
  Protokolsüz: 13260

PHISHING (25000 toplam):
  https://  : 900
  http://   : 11286
  Protokolsüz: 12803


In [3]:
import os
for file in os.listdir('data/raw/'):
    print(file)

kaggle_phishing.csv
phishing_site_urls.csv
PhiUSIIL_Phishing_URL_Dataset.csv
raw_dataset.csv
urldata.csv


In [4]:
phiusiil = pd.read_csv('data/raw/PhiUSIIL_Phishing_URL_Dataset.csv')
print(f"Toplam satır: {len(phiusiil)}")
print(f"Kolonlar: {phiusiil.columns.tolist()}")
print(f"\nİlk 5 satır:")
print(phiusiil.head())

Toplam satır: 235795
Kolonlar: ['FILENAME', 'URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']

İlk 5 satır:
     FILENAME                       

In [5]:
print(f"Label dağılımı:")
print(phiusiil['label'].value_counts())

print(f"\nIsHTTPS dağılımı:")
print(phiusiil['IsHTTPS'].value_counts())

print(f"\nBizim için sadece URL ve label lazım:")
print(phiusiil[['URL', 'label']].head(10))

Label dağılımı:
label
1    134850
0    100945
Name: count, dtype: int64

IsHTTPS dağılımı:
IsHTTPS
1    184539
0     51256
Name: count, dtype: int64

Bizim için sadece URL ve label lazım:
                                  URL  label
0    https://www.southbankmosaics.com      1
1            https://www.uni-mainz.de      1
2      https://www.voicefmradio.co.uk      1
3         https://www.sfnmjournal.com      1
4  https://www.rewildingargentina.org      1
5     https://www.globalreporting.org      1
6          https://www.saffronart.com      1
7          https://www.nerdscandy.com      1
8      https://www.hyderabadonline.in      1
9                 https://www.aap.org      1


In [6]:
from datetime import datetime

# Label'ları düzelt (1→0 meşru, 0→1 phishing)
phiusiil['label_fixed'] = phiusiil['label'].apply(lambda x: 0 if x == 1 else 1)

print(f"Düzeltilmiş label dağılımı:")
print(phiusiil['label_fixed'].value_counts())

# HTTPS dağılımına göre ihtiyacımız olanları seç
# 1. HTTP meşru siteler (bizde çok az vardı)
http_mesru = phiusiil[(phiusiil['IsHTTPS'] == 0) & (phiusiil['label_fixed'] == 0)]['URL']
# 2. HTTPS phishing siteler (bizde çok az vardı)
https_phishing = phiusiil[(phiusiil['IsHTTPS'] == 1) & (phiusiil['label_fixed'] == 1)]['URL']

print(f"\nPhiUSIIL'de HTTP meşru: {len(http_mesru)}")
print(f"PhiUSIIL'de HTTPS phishing: {len(https_phishing)}")

Düzeltilmiş label dağılımı:
label_fixed
0    134850
1    100945
Name: count, dtype: int64

PhiUSIIL'de HTTP meşru: 0
PhiUSIIL'de HTTPS phishing: 49689


In [7]:
import requests
import zipfile
import io

# 1. HTTPS phishing - PhiUSIIL'den al
https_ph_sample = https_phishing.sample(10000, random_state=42)
df_https_ph = pd.DataFrame({
    'url': https_ph_sample.values,
    'timestamp': datetime.now(),
    'label': 1,
    'source': 'phiusiil'
})

print(f"HTTPS phishing eklenecek: {len(df_https_ph)}")

# 2. HTTP meşru - Tranco'dan al
print("Tranco indiriliyor...")
r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])
top_sites = tranco[tranco['rank'] <= 10000]['domain'].tolist()

http_mesru_urls = [f"http://{domain}" for domain in top_sites]
df_http_mesru = pd.DataFrame({
    'url': http_mesru_urls,
    'timestamp': datetime.now(),
    'label': 0,
    'source': 'tranco_http'
})

print(f"HTTP meşru eklenecek: {len(df_http_mesru)}")
print(f"\nToplam eklenecek: {len(df_https_ph) + len(df_http_mesru)}")

HTTPS phishing eklenecek: 10000
Tranco indiriliyor...
HTTP meşru eklenecek: 10000

Toplam eklenecek: 20000


In [8]:
# İkisini birleştir
df_new = pd.concat([df_https_ph, df_http_mesru], ignore_index=True)
print(f"Toplam yeni URL: {len(df_new)}")
print(f"Label dağılımı: {df_new['label'].value_counts().to_dict()}")

print("\nFeature extraction başlıyor, 20-30 dakika sürebilir...")
new_features = pipeline.transform(df_new, verbose=True)
print(f"Tamamlandı! Shape: {new_features.shape}")

Toplam yeni URL: 20000
Label dağılımı: {1: 10000, 0: 10000}

Feature extraction başlıyor, 20-30 dakika sürebilir...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████| 20000/20000 [00:01<00:00, 16795.79it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████| 20000/20000 [00:51<00:00, 385.62it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████| 20000/20000 [00:00<00:00, 22200.31it/s]


Feature extraction tamamlandi: 20000 satir, 64 feature
Tamamlandı! Shape: (20000, 67)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

# Yeni feature'ları hazırla
feature_cols = [c for c in new_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_new = new_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_new = df_new['label'].values

# Mevcut train seti ile birleştir
X_train_combined = np.vstack([X_train, X_new])
y_train_combined = np.concatenate([y_train, y_new])

print(f"Eski train: {X_train.shape}")
print(f"Yeni train: {X_train_combined.shape}")
print(f"Label dağılımı: {pd.Series(y_train_combined).value_counts().to_dict()}")

# Cross-validation ile eğit
print("\nCross-validation yapılıyor...")
rf_new = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

cv_scores = cross_val_score(rf_new, X_train_combined, y_train_combined, cv=5, scoring='accuracy')
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Final model eğit
print("\nFinal model eğitiliyor...")
rf_new.fit(X_train_combined, y_train_combined)

# Test
y_pred = rf_new.predict(X_test)
y_prob = rf_new.predict_proba(X_test)[:,1]
train_acc = accuracy_score(y_train_combined, rf_new.predict(X_train_combined))
test_acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nSonuçlar:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Fark (Overfitting): {train_acc - test_acc:.4f}")
print(f"F1: {f1:.4f}")
print(f"AUC: {auc:.4f}")

Eski train: (24987, 64)
Yeni train: (44987, 64)
Label dağılımı: {0: 22552, 1: 22435}

Cross-validation yapılıyor...
5-Fold CV Accuracy: 0.9543 ± 0.0318

Final model eğitiliyor...

Sonuçlar:
Train Accuracy: 0.9839
Test Accuracy:  0.9284
Fark (Overfitting): 0.0555
F1: 0.9287
AUC: 0.9801


In [10]:
test_urls = [
    # HTTP meşru
    "http://bbc.com",
    "http://cnn.com",
    "http://wikipedia.org",
    # HTTPS phishing
    "https://amazon.com-prime-offer.net",
    "https://paypal-secure-login-verify.com",
    "https://google-account-suspended.com",
    # Klasik phishing
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("Test:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_new.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Test:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://bbc.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://cnn.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://wikipedia.org

 Feature extraction basliyor...
   [1/4] URL features

In [11]:
http_phishing = phiusiil[(phiusiil['IsHTTPS'] == 0) & (phiusiil['label_fixed'] == 1)]['URL']
print(f"PhiUSIIL'de HTTP phishing: {len(http_phishing)}")

PhiUSIIL'de HTTP phishing: 51256


In [12]:
# HTTP phishing ekle - dengelemek için 10000 al
http_ph_sample = http_phishing.sample(10000, random_state=42)
df_http_ph = pd.DataFrame({
    'url': http_ph_sample.values,
    'timestamp': datetime.now(),
    'label': 1,
    'source': 'phiusiil_http'
})

print(f"HTTP phishing eklenecek: {len(df_http_ph)}")
print("Feature extraction başlıyor...")

http_ph_features = pipeline.transform(df_http_ph, verbose=True)
print(f"Tamamlandı! Shape: {http_ph_features.shape}")

HTTP phishing eklenecek: 10000
Feature extraction başlıyor...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 17302.51it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████| 10000/10000 [00:31<00:00, 320.11it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 21765.53it/s]

Feature extraction tamamlandi: 10000 satir, 64 feature
Tamamlandı! Shape: (10000, 67)


In [13]:
# HTTP phishing feature'larını hazırla
feature_cols = [c for c in http_ph_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_http_ph = http_ph_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_http_ph = np.ones(len(X_http_ph))

# Hepsini birleştir
X_train_final = np.vstack([X_train_combined, X_http_ph])
y_train_final = np.concatenate([y_train_combined, y_http_ph])

print(f"Önceki train: {X_train_combined.shape}")
print(f"Final train: {X_train_final.shape}")
print(f"Label dağılımı: {pd.Series(y_train_final).value_counts().to_dict()}")

# Yeniden eğit
print("\nModel eğitiliyor...")
rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X_train_final, y_train_final)

# Test
y_pred = rf_final.predict(X_test)
y_prob = rf_final.predict_proba(X_test)[:,1]
train_acc = accuracy_score(y_train_final, rf_final.predict(X_train_final))
test_acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nSonuçlar:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Overfitting Farkı: {train_acc - test_acc:.4f}")
print(f"F1: {f1:.4f}")
print(f"AUC: {auc:.4f}")

Önceki train: (44987, 64)
Final train: (54987, 64)
Label dağılımı: {1.0: 32435, 0.0: 22552}

Model eğitiliyor...

Sonuçlar:
Train Accuracy: 0.9839
Test Accuracy:  0.9271
Overfitting Farkı: 0.0568
F1: 0.9276
AUC: 0.9801


In [14]:
test_urls = [
    # HTTP meşru
    "http://bbc.com",
    "http://cnn.com",
    "http://wikipedia.org",
    # HTTPS phishing
    "https://amazon.com-prime-offer.net",
    "https://paypal-secure-login-verify.com",
    "https://google-account-suspended.com",
    # HTTP phishing
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
    # Klasik meşru
    "https://google.com",
    "https://gov.tr",
    "https://onedio.com",
]

print("Final Model Testi:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_final.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Final Model Testi:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://bbc.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://cnn.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.02) → http://wikipedia.org

 Feature extraction basliyor...
   [1/4]

In [15]:
# Mevcut dağılımı gör
print(f"Phishing: {(y_train_final == 1).sum()}")
print(f"Meşru: {(y_train_final == 0).sum()}")
print(f"Fark: {(y_train_final == 1).sum() - (y_train_final == 0).sum()}")
print(f"\nDengelemek için {(y_train_final == 1).sum() - (y_train_final == 0).sum()} meşru URL daha eklemek lazım")

Phishing: 32435
Meşru: 22552
Fark: 9883

Dengelemek için 9883 meşru URL daha eklemek lazım


In [16]:
# Tranco'dan 10K daha meşru ekle (farklı siteler, rank 10001-20000)
top_sites_2 = tranco[(tranco['rank'] > 10000) & (tranco['rank'] <= 20000)]['domain'].tolist()

extra_mesru_urls = [f"https://{domain}" for domain in top_sites_2]
df_extra_mesru = pd.DataFrame({
    'url': extra_mesru_urls,
    'timestamp': datetime.now(),
    'label': 0,
    'source': 'tranco_extra'
})

print(f"Eklenecek meşru URL: {len(df_extra_mesru)}")
print("Feature extraction başlıyor...")

extra_features = pipeline.transform(df_extra_mesru, verbose=True)
print(f"Tamamlandı!")

Eklenecek meşru URL: 10000
Feature extraction başlıyor...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 22831.86it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████| 10000/10000 [00:26<00:00, 371.38it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 37796.50it/s]

Feature extraction tamamlandi: 10000 satir, 64 feature
Tamamlandı!


In [17]:
# Extra meşru feature'larını hazırla
feature_cols = [c for c in extra_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X_extra = extra_features[feature_cols].reindex(columns=feature_names, fill_value=0).values
y_extra = np.zeros(len(X_extra))

# Hepsini birleştir
X_train_balanced = np.vstack([X_train_final, X_extra])
y_train_balanced = np.concatenate([y_train_final, y_extra])

print(f"Final train: {X_train_balanced.shape}")
print(f"Label dağılımı: {pd.Series(y_train_balanced).value_counts().to_dict()}")

# Eğit
print("\nModel eğitiliyor...")
rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf_balanced.fit(X_train_balanced, y_train_balanced)

# Test
y_pred = rf_balanced.predict(X_test)
y_prob = rf_balanced.predict_proba(X_test)[:,1]
train_acc = accuracy_score(y_train_balanced, rf_balanced.predict(X_train_balanced))
test_acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"\nSonuçlar:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Overfitting Farkı: {train_acc - test_acc:.4f}")
print(f"F1: {f1:.4f}")
print(f"AUC: {auc:.4f}")

Final train: (64987, 64)
Label dağılımı: {0.0: 32552, 1.0: 32435}

Model eğitiliyor...

Sonuçlar:
Train Accuracy: 0.9860
Test Accuracy:  0.9281
Overfitting Farkı: 0.0579
F1: 0.9286
AUC: 0.9802


In [18]:
test_urls = [
    "http://bbc.com",
    "http://cnn.com",
    "https://google.com",
    "https://gov.tr",
    "https://onedio.com",
    "https://amazon.com-prime-offer.net",
    "https://paypal-secure-login-verify.com",
    "http://paypal-login-verify.xyz",
    "http://amazon-secure-update.tk",
]

print("Dengeli Model Testi:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_balanced.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Dengeli Model Testi:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://bbc.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → http://cnn.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.00) → https://google.com

 Feature extraction basliyor...
   [1/4]

In [19]:
joblib.dump(rf_balanced, 'data/models/baseline_random_forest.pkl')
print("✅ Model kaydedildi!")

✅ Model kaydedildi!


In [20]:
import numpy as np

# Cialdini feature'ları hangileri?
cialdini_features = [f for f in feature_names if any(x in f.lower() for x in 
    ['authority', 'scarcity', 'fear', 'reciprocity', 'social', 'commitment', 
     'cialdini', 'urgency', 'reward', 'proof'])]

print("Cialdini feature'ları ve ağırlıkları:")
print("-" * 50)
for f in cialdini_features:
    idx = feature_names.index(f)
    print(f"{f:40} {rf_balanced.feature_importances_[idx]:.4f}")

print(f"\nToplam Cialdini ağırlığı: {sum(rf_balanced.feature_importances_[feature_names.index(f)] for f in cialdini_features):.4f}")
print(f"Toplam diğer ağırlık: {1 - sum(rf_balanced.feature_importances_[feature_names.index(f)] for f in cialdini_features):.4f}")

Cialdini feature'ları ve ağırlıkları:
--------------------------------------------------
cld_authority                            0.0029
cld_scarcity                             0.0002
cld_fear                                 0.0000
cld_reciprocity                          0.0002
cld_social_proof                         0.0011
cld_commitment                           0.0073

Toplam Cialdini ağırlığı: 0.0116
Toplam diğer ağırlık: 0.9884


In [21]:
# Borderline URL analizi
borderline_urls = [
    "https://paypal-secure-login-verify.com",
    "http://amazon-secure-update.tk",
]

for url in borderline_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = rf_balanced.predict_proba(X)[0][1]
    
    # En etkili feature'lar
    importances = rf_balanced.feature_importances_
    top_idx = np.argsort(importances)[::-1][:10]
    
    print(f"\nURL: {url}")
    print(f"Prob: {prob:.4f}")
    print("En etkili 10 feature:")
    for i in top_idx:
        if i < X.shape[1]:
            print(f"  {feature_names[i]:35} değer={X[0][i]:.3f} önem={importances[i]:.4f}")


 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature

URL: https://paypal-secure-login-verify.com
Prob: 0.5284
En etkili 10 feature:
  subdomain_length                    değer=0.000 önem=0.1026
  num_dots                            değer=1.000 önem=0.0925
  path_length                         değer=0.000 önem=0.0784
  num_subdomains                      değer=0.000 önem=0.0765
  num_slashes                         değer=2.000 önem=0.0757
  url_length                          değer=38.000 önem=0.0679
  path_entropy                        değer=0.000 önem=0.0468
  path_depth                          değer=0.000 önem=0.0391
  is_https                            değer=1.000 önem=0.0384
  tld_trusted                         değer=1.000 önem=0.0362

 Feature extraction basliyor...
   [1/4

In [22]:
# Domain ile ilgili feature'ları göster
domain_features = [f for f in feature_names if any(x in f.lower() for x in 
    ['domain', 'hyphen', 'brand', 'levenshtein'])]

print("Domain feature'ları:")
for f in domain_features:
    idx = feature_names.index(f)
    print(f"{f:40} önem={rf_balanced.feature_importances_[idx]:.4f}")

Domain feature'ları:
domain_length                            önem=0.0156
subdomain_length                         önem=0.1026
num_subdomains                           önem=0.0765
num_hyphens                              önem=0.0147
domain_entropy                           önem=0.0152
domain_has_digit                         önem=0.0024
domain_has_hyphen                        önem=0.0016
min_brand_levenshtein                    önem=0.0171
min_brand_levenshtein_norm               önem=0.0137
is_near_brand                            önem=0.0010
is_exact_brand                           önem=0.0033
brand_in_non_brand_domain                önem=0.0148
domain_confusion_score                   önem=0.0223


In [23]:
# Bu URL'deki domain feature değerleri
url = "https://paypal-secure-login-verify.com"
df_test = pd.DataFrame({'url': [url], 'label': [0]})
features = pipeline.transform(df_test, verbose=False)
feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values

print(f"URL: {url}")
print("\nDomain feature değerleri:")
for f in domain_features:
    idx = feature_names.index(f)
    print(f"  {f:40} değer={X[0][idx]:.4f}")


 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
URL: https://paypal-secure-login-verify.com

Domain feature değerleri:
  domain_length                            değer=26.0000
  subdomain_length                         değer=0.0000
  num_subdomains                           değer=0.0000
  num_hyphens                              değer=3.0000
  domain_entropy                           değer=3.8731
  domain_has_digit                         değer=0.0000
  domain_has_hyphen                        değer=1.0000
  min_brand_levenshtein                    değer=19.0000
  min_brand_levenshtein_norm               değer=0.7308
  is_near_brand                            değer=0.0000
  is_exact_brand                           değer=0.0000
  brand_in_non_brand_domain                değer=1.0

In [24]:
with open('src/feature_extractor.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Phishing domain skoru hesaplayan yeni feature ekle
old = "features[\"domain_confusion_score\"]"
idx = content.find(old)
print(f"domain_confusion_score satırı bulundu: {idx}")
print(content[idx:idx+200])

domain_confusion_score satırı bulundu: 8819
features["domain_confusion_score"]   = self._confusion_score(domain, lev_scores)
        except Exception:
            features = {
                "min_brand_levenshtein": 999,
                "min_b


In [25]:
# _confusion_score fonksiyonunu göster
idx = content.find('def _confusion_score')
print(content[idx:idx+500])

def _confusion_score(self, domain: str, lev_scores: Dict) -> float:
        score = 0.0
        if lev_scores["min_dist"] == 1:
            score += 0.5
        elif lev_scores["min_dist"] == 2:
            score += 0.3
        score += min(0.3, self._count_homoglyphs(domain) * 0.1)
        # Brand domain içinde geçiyorsa yüksek skor
        if self._brand_in_domain(domain):
            score += 0.4
        # Tire sayısı fazlaysa şüpheli
        hyphen_count = domain.count("-")
        if hyphen


In [26]:
old_func = '''def _confusion_score(self, domain: str, lev_scores: Dict) -> float:
        score = 0.0
        if lev_scores["min_dist"] == 1:
            score += 0.5
        elif lev_scores["min_dist"] == 2:
            score += 0.3
        score += min(0.3, self._count_homoglyphs(domain) * 0.1)
        score += 0.2 if self._brand_in_domain(domain) else 0.0
        return round(min(score, 1.0), 4)'''

new_func = '''def _confusion_score(self, domain: str, lev_scores: Dict) -> float:
        score = 0.0
        if lev_scores["min_dist"] == 1:
            score += 0.5
        elif lev_scores["min_dist"] == 2:
            score += 0.3
        score += min(0.3, self._count_homoglyphs(domain) * 0.1)
        # Brand domain içinde geçiyorsa yüksek skor
        if self._brand_in_domain(domain):
            score += 0.4
        # Tire sayısı fazlaysa şüpheli
        hyphen_count = domain.count("-")
        if hyphen_count >= 3:
            score += 0.4
        elif hyphen_count == 2:
            score += 0.2
        elif hyphen_count == 1:
            score += 0.1
        # Domain çok uzunsa şüpheli
        if len(domain) > 20:
            score += 0.2
        elif len(domain) > 15:
            score += 0.1
        return round(min(score, 1.0), 4)'''

content = content.replace(old_func, new_func)

with open('src/feature_extractor.py', 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ _confusion_score güncellendi!")

✅ _confusion_score güncellendi!


In [27]:
import sys
import os
sys.path.append(r'C:\Users\emira\OneDrive\Desktop\main projem')
os.chdir(r'C:\Users\emira\OneDrive\Desktop\main projem')

import numpy as np
import pandas as pd
import json
import joblib
from src.feature_extractor import FeaturePipeline

model = joblib.load('data/models/baseline_random_forest.pkl')
with open('data/processed/feature_names.json') as f:
    feature_names = json.load(f)
pipeline = FeaturePipeline()

print("Yüklendi!")

Yüklendi!


In [28]:
test_urls = [
    "https://paypal-secure-login-verify.com",
    "http://amazon-secure-update.tk",
    "https://amazon.com-prime-offer.net",
    "https://google.com",
    "https://gov.tr",
    "https://onedio.com",
    "http://bbc.com",
    "http://paypal-login-verify.xyz",
]

print("Güncellenen confusion_score ile test:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Güncellenen confusion_score ile test:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.53) → https://paypal-secure-login-verify.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
✅ MEŞRU (0.45) → http://amazon-secure-update.tk

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.82)

In [29]:
import requests
import zipfile
import io
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

# Tüm dataseti yükle
df_original = pd.read_csv('data/processed/raw_dataset.csv')

# PhiUSIIL'den eklediklerimizi de ekle
phiusiil = pd.read_csv('data/raw/PhiUSIIL_Phishing_URL_Dataset.csv')
phiusiil['label_fixed'] = phiusiil['label'].apply(lambda x: 0 if x == 1 else 1)

https_phishing = phiusiil[(phiusiil['IsHTTPS'] == 1) & (phiusiil['label_fixed'] == 1)]['URL']
http_phishing = phiusiil[(phiusiil['IsHTTPS'] == 0) & (phiusiil['label_fixed'] == 1)]['URL']

https_ph_sample = https_phishing.sample(10000, random_state=42)
http_ph_sample = http_phishing.sample(10000, random_state=42)

# Tranco
print("Tranco indiriliyor...")
r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=30)
z = zipfile.ZipFile(io.BytesIO(r.content))
tranco = pd.read_csv(z.open(z.namelist()[0]), header=None, names=['rank', 'domain'])

http_mesru = [f"http://{d}" for d in tranco[tranco['rank'] <= 10000]['domain'].tolist()]
https_mesru = [f"https://{d}" for d in tranco[(tranco['rank'] > 10000) & (tranco['rank'] <= 20000)]['domain'].tolist()]

# Hepsini birleştir
df_all = pd.concat([
    df_original,
    pd.DataFrame({'url': https_ph_sample.values, 'timestamp': datetime.now(), 'label': 1, 'source': 'phiusiil_https'}),
    pd.DataFrame({'url': http_ph_sample.values, 'timestamp': datetime.now(), 'label': 1, 'source': 'phiusiil_http'}),
    pd.DataFrame({'url': http_mesru, 'timestamp': datetime.now(), 'label': 0, 'source': 'tranco_http'}),
    pd.DataFrame({'url': https_mesru, 'timestamp': datetime.now(), 'label': 0, 'source': 'tranco_https'}),
], ignore_index=True)

print(f"Toplam URL: {len(df_all)}")
print(f"Label dağılımı: {df_all['label'].value_counts().to_dict()}")

Tranco indiriliyor...
Toplam URL: 90000
Label dağılımı: {0: 45000, 1: 45000}


In [30]:
print("Feature extraction başlıyor...")
print("Bu 1-2 saat sürebilir, bilgisayarı kapatma!")

all_features = pipeline.transform(df_all, verbose=True)
print(f"\nTamamlandı! Shape: {all_features.shape}")

Feature extraction başlıyor...
Bu 1-2 saat sürebilir, bilgisayarı kapatma!

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████| 90000/90000 [00:05<00:00, 15369.09it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████| 90000/90000 [04:46<00:00, 314.63it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████| 90000/90000 [00:05<00:00, 15964.20it/s]


Feature extraction tamamlandi: 90000 satir, 64 feature

Tamamlandı! Shape: (90000, 67)


In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

# Feature ve label ayır
feature_cols = [c for c in all_features.columns if c not in ['url', 'label', 'timestamp', 'source']]
X = all_features[feature_cols].values
y = all_features['label'].values

# Feature isimlerini kaydet
import json
with open('data/processed/feature_names.json', 'w') as f:
    json.dump(feature_cols, f)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Model eğitiliyor...")

# Model eğit
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Değerlendir
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(f"\n✅ Sonuçlar:")
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"  F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"  ROC AUC  : {roc_auc_score(y_test, y_prob):.4f}")

# Kaydet
joblib.dump(rf, 'data/models/baseline_random_forest.pkl')
print("\n✅ Model kaydedildi!")

Train: (72000, 64), Test: (18000, 64)
Model eğitiliyor...

✅ Sonuçlar:
  Accuracy : 0.9558
  F1 Score : 0.9555
  ROC AUC  : 0.9912

✅ Model kaydedildi!


In [32]:
# Kernel restart YAPMA, direkt çalıştır
model = joblib.load('data/models/baseline_random_forest.pkl')
with open('data/processed/feature_names.json') as f:
    feature_names = json.load(f)

test_urls = [
    "https://paypal-secure-login-verify.com",
    "http://amazon-secure-update.tk",
    "https://amazon.com-prime-offer.net",
    "https://google.com",
    "https://gov.tr",
    "https://onedio.com",
    "http://bbc.com",
    "http://paypal-login-verify.xyz",
]

print("Yeni model ile test:")
print("-" * 60)
for url in test_urls:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp', 'source']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    print(f"{pred} ({prob:.2f}) → {url}")

Yeni model ile test:
------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.60) → https://paypal-secure-login-verify.com

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.63) → http://amazon-secure-update.tk

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.81) → https://ama

In [33]:
zor_mesru = [
    "http://v2.wp-api.org/endpoints/posts/index.html?id=123",
    "http://archive.ubuntu.com/ubuntu/dists/trusty/main/binary-amd64/Packages.gz",
    "http://www.columbia.edu/~fdc/sample.html?query=test-data",
    "http://daringfireball.net/projects/markdown/syntax#link",
    "http://mirrors.edge.kernel.org/pub/linux/kernel/v5.x/ChangeLog-5.0.1",
    "http://info.cern.ch/hypertext/WWW/TheProject.html",
    "http://lib.metu.edu.tr/search~S1?/atest/atest/1%2C130%2C130%2CB/frameset&FF=atest&1%2C%2C4",
    "http://api.openweathermap.org/data/2.5/weather?q=Istanbul&appid=12345",
    "http://www.isoc.org/internet/history/brief.shtml#Origin",
    "http://ftp.gnu.org/gnu/grep/grep-3.4.tar.xz.sig",
]

print("Zor meşru HTTP URL'leri testi:")
print("-" * 70)
dogru = 0
for url in zor_mesru:
    df_test = pd.DataFrame({'url': [url], 'label': [0]})
    features = pipeline.transform(df_test, verbose=False)
    feature_cols = [c for c in features.columns if c not in ['url', 'label', 'timestamp', 'source']]
    X = features[feature_cols].reindex(columns=feature_names, fill_value=0).values
    prob = model.predict_proba(X)[0][1]
    pred = "🔴 PHİSHİNG" if prob > 0.50 else "✅ MEŞRU"
    dogru_mu = "✓" if prob <= 0.50 else "✗ YANLIŞ"
    if prob <= 0.50:
        dogru += 1
    print(f"{pred} ({prob:.2f}) {dogru_mu} → {url[:60]}")

print(f"\nSonuç: {dogru}/{len(zor_mesru)} doğru ({dogru/len(zor_mesru)*100:.0f}%)")

Zor meşru HTTP URL'leri testi:
----------------------------------------------------------------------

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.83) ✗ YANLIŞ → http://v2.wp-api.org/endpoints/posts/index.html?id=123

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...
Feature extraction tamamlandi: 1 satir, 64 feature
🔴 PHİSHİNG (0.63) ✗ YANLIŞ → http://archive.ubuntu.com/ubuntu/dists/trusty/main/binary-am

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...
   [2/4] Adversarial features cikariliyor...
   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor..

In [34]:
import requests
import pandas as pd
import zipfile
import io

# Kaggle'dan hazır phishing dataset - dışarıdan erişilebilir
# Web Page Phishing Detection Dataset
url = "https://raw.githubusercontent.com/GregaVrbancic/Phishing-Dataset/master/dataset_full.csv"

print("Dataset indiriliyor...")
df_ext = pd.read_csv(url)
print(f"Shape: {df_ext.shape}")
print(df_ext.head(3))
print(df_ext.columns.tolist())

Dataset indiriliyor...
Shape: (88647, 112)
   qty_dot_url  qty_hyphen_url  qty_underline_url  qty_slash_url  \
0            3               0                  0              1   
1            5               0                  1              3   
2            2               0                  0              1   

   qty_questionmark_url  qty_equal_url  qty_at_url  qty_and_url  \
0                     0              0           0            0   
1                     0              3           0            2   
2                     0              0           0            0   

   qty_exclamation_url  qty_space_url  ...  qty_ip_resolved  qty_nameservers  \
0                    0              0  ...                1                2   
1                    0              0  ...                1                2   
2                    0              0  ...                1                2   

   qty_mx_servers  ttl_hostname  tls_ssl_certificate  qty_redirects  \
0               0      

In [35]:
# PhishTank + DMOZ/Alexa URL'leri içeren dataset
urls_dataset = "https://raw.githubusercontent.com/datasets/phishing-urls/main/data/phishing_urls.csv"

try:
    df_urls = pd.read_csv(urls_dataset)
    print(f"Shape: {df_urls.shape}")
    print(df_urls.head(3))
    print(df_urls.columns.tolist())
except:
    print("Bu çalışmadı, alternatif deniyoruz...")
    
    # Alternatif 2
    url2 = "https://raw.githubusercontent.com/faizann24/Using-machine-learning-to-detect-malicious-URLs/master/data/data.csv"
    df_urls = pd.read_csv(url2)
    print(f"Shape: {df_urls.shape}")
    print(df_urls.head(3))
    print(df_urls.columns.tolist())

Bu çalışmadı, alternatif deniyoruz...
Shape: (420464, 2)
                      url label
0  diaryofagameaddict.com   bad
1        espdesign.com.au   bad
2      iamagameaddict.com   bad
['url', 'label']


In [36]:
# Label dağılımına bak
print(df_urls['label'].value_counts())
print(f"\nToplam: {len(df_urls)}")

label
good    344821
bad      75643
Name: count, dtype: int64

Toplam: 420464


In [37]:
url2 = "https://raw.githubusercontent.com/faizann24/Using-machine-learning-to-detect-malicious-URLs/master/data/data.csv"
df_urls = pd.read_csv(url2)

good_sample = df_urls[df_urls['label'] == 'good'].sample(200, random_state=42)
bad_sample = df_urls[df_urls['label'] == 'bad'].sample(200, random_state=42)

df_test_ext = pd.concat([good_sample, bad_sample], ignore_index=True)
df_test_ext['label_num'] = df_test_ext['label'].map({'good': 0, 'bad': 1})
df_test_ext['url_full'] = df_test_ext['url'].apply(
    lambda x: x if x.startswith('http') else f'http://{x}'
)
print("Hazır!")

Hazır!


In [38]:
df_input = pd.DataFrame({
    'url': df_test_ext['url_full'].values,
    'label': df_test_ext['label_num'].values
})

print(f"Test seti: {len(df_input)} URL")
print("Feature extraction başlıyor...")
features_ext = pipeline.transform(df_input, verbose=True)
print(f"Tamamlandı! Shape: {features_ext.shape}")

Test seti: 400 URL
Feature extraction başlıyor...

 Feature extraction basliyor...
   [1/4] URL features cikariliyor...


URL Features: 100%|████████████████████████████████████████████████████████████████████████| 400/400 [00:00<00:00, 16641.09it/s]


   [2/4] Adversarial features cikariliyor...


Adversarial Features: 100%|██████████████████████████████████████████████████████████████████| 400/400 [00:01<00:00, 344.19it/s]


   [3/4] Metadata features isleniyor...
   [4/4] Cialdini psikolojik features cikariliyor...


Cialdini Features: 100%|███████████████████████████████████████████████████████████████████| 400/400 [00:00<00:00, 17904.48it/s]

Feature extraction tamamlandi: 400 satir, 64 feature
Tamamlandı! Shape: (400, 66)
